# 02_screening
Jalankan **dari atas ke bawah**. Sel 1 dan Sel 2 adalah SEL PEMBUKA: wajib dijalankan setiap kali membuka notebook.

## Sel 1 — SEL PEMBUKA (Python): Drive, R, dan pustaka sistem

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%load_ext rpy2.ipython
!apt-get -qq update > /dev/null
!apt-get -qq install -y libpoppler-cpp-dev libglpk-dev libxml2-dev libcurl4-openssl-dev libssl-dev libfontconfig1-dev libharfbuzz-dev libfribidi-dev libfreetype6-dev libpng-dev libtiff5-dev libjpeg-dev > /dev/null
print('Sel 1 selesai')

## Sel 2 — SEL PEMBUKA (R): paket dan folder kerja

In [ ]:
%%R
proj <- "/content/drive/MyDrive/bibliometrik-cxr"
lib  <- file.path(proj, "Rlib")
dir.create(lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(lib, .libPaths()))
codename <- system("lsb_release -cs", intern = TRUE)
options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", codename)))
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
  paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
pkgs <- c("bibliometrix", "UpSetR", "readxl", "writexl", "ggplot2", "htmlwidgets")
baru <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(baru)) install.packages(baru, lib = lib)
setwd(proj)
suppressPackageStartupMessages(library(bibliometrix))
cat("Folder kerja:", getwd(), "\nbibliometrix", as.character(packageVersion("bibliometrix")), "siap\n")

## Sel 3 — Langkah 8: buat lembar screening
Aman dijalankan ulang: jika `screening.xlsx` sudah ada, file itu **tidak** ditimpa.

In [ ]:
%%R
library(writexl)
files <- list.files("02_raw", pattern = "^scopus_S1.*\\.csv$", full.names = TRUE)
print(files)
raw <- do.call(rbind, lapply(files, read.csv, check.names = FALSE, fileEncoding = "UTF-8-BOM"))
raw <- raw[!duplicated(raw$EID), ]
saveRDS(raw, "03_clean/raw.rds")
if (!file.exists("03_clean/screening.xlsx")) {
  scr <- raw[, c("EID", "Title", "Abstract", "Year", "Source title", "Document Type", "DOI")]
  scr$keputusan <- ""; scr$alasan <- ""
  write_xlsx(scr, "03_clean/screening.xlsx")
  cat("screening.xlsx baru dibuat\n")
} else cat("screening.xlsx sudah ada, tidak ditimpa\n")
cat("Jumlah dokumen:", nrow(raw), "| Ada kolom References:", "References" %in% names(raw), "\n")

In [ ]:
%%R
print(list.files("03_clean"))
print(list.files(".", pattern = "screening", recursive = TRUE))

In [ ]:
%%R
library(readxl)
h <- read_excel("03_clean/screening.xlsx")
print(names(h))
print(table(h$keputusan))

## Sel 4 — Langkah 11: saring corpus dan siapkan semua file analisis

In [ ]:
%%R
library(readxl); library(bibliometrix)
hasil <- read_excel("03_clean/screening.xlsx")
raw   <- readRDS("03_clean/raw.rds")

# Angka untuk diagram PRISMA
print(table(hasil$keputusan))
print(table(hasil$alasan))

# Corpus bersih
keep  <- hasil$EID[hasil$keputusan == "I"]
clean <- raw[raw$EID %in% keep, ]
write.csv(clean, "03_clean/corpus_S1_clean.csv", row.names = FALSE, fileEncoding = "UTF-8")
write.csv(clean[clean$Year >= 2022, ], "03_clean/corpus_2022_2026.csv",
          row.names = FALSE, fileEncoding = "UTF-8")

# Jenis tugas per dokumen (dipakai di Tahap F)
write.csv(hasil[hasil$keputusan == "I", c("EID", "Title", "Year", "tugas")],
          "03_clean/tugas.csv", row.names = FALSE, fileEncoding = "UTF-8")

# Format bibliometrix
M <- convert2df("03_clean/corpus_S1_clean.csv", dbsource = "scopus", format = "csv")
saveRDS(M, "03_clean/M.rds")

# Thesaurus VOSviewer (kolom kanan kosong = istilah dibuang)
thes <- data.frame(
  label = c("pseudo-labelling", "pseudo label", "pseudo-label", "cxr",
            "chest radiograph", "chest x-rays", "semi supervised learning",
            "imbalanced data", "data imbalance", "multilabel classification",
            "deep learning", "machine learning"),
  replace = c("pseudo-labeling", "pseudo-labeling", "pseudo-labeling",
              "chest x-ray", "chest x-ray", "chest x-ray", "semi-supervised learning",
              "class imbalance", "class imbalance", "multi-label classification",
              "", ""))
names(thes) <- c("label", "replace by")
write.table(thes, "03_clean/thesaurus_vos.txt", sep = "\t", row.names = FALSE, quote = FALSE)

cat("Corpus bersih:", nrow(clean), "| 2022-2026:", sum(clean$Year >= 2022),
    "| M:", nrow(M), "baris\n")
list.files("03_clean")